<a href="https://colab.research.google.com/github/laugarcias/FIAP--3/blob/main/C%C3%B3pia_de_aula_spark_hadson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalando pacote PySpark
!pip install pyspark

In [ ]:
#Instalação de funções básicas para o Spark
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct

Iniciando Ambiente Spark e Criando a Sessão SparkSession

In [ ]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .getOrCreate()


)

In [ ]:
print(spark.version)

4.0.4


Importação de Bases.csv

In [ ]:
df_clientes = spark.read.csv('clientes.csv', sep = ";", inferSchema= True, header = True)
df_vendas = spark.read.csv('vendas.csv', sep = ",", inferSchema= True, header = True)

In [ ]:
df_clientes.show()
df_vendas.show()

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         5|    Guilherme Barros|davi-luizfarias@b...|   24|              Cuiaba| MT| Feminino|   20/04/2021|
|         6|Sr. Guilherme Rez...|cavalcantieloah@b...|   57|Sao Bernardo do C...| SP|Masculino|   13/08/2023|
|         

Exemplo de RDD

In [ ]:
dados_clientes = [
    (1,"Laura Garcias", "barretolaura775@gmail.com", 24,"São Pualo", "SP"),
    (2,"Marcos Viniciues da mata Ribeiro", "damatamarcos@gmail.com",28,"Curitiba","PR")
]

Não é utilizado mais RDD - esta "obsoleto" somente para momentos específicos

In [ ]:
rdd_clientes = spark.sparkContext.parallelize(dados_clientes)

In [ ]:
rdd_clientes.collect()

[(1, 'Laura Garcias', 'barretolaura775@gmail.com', 24, 'São Pualo', 'SP'),
 (2,
  'Marcos Viniciues da mata Ribeiro',
  'damatamarcos@gmail.com',
  28,
  'Curitiba',
  'PR')]

Exploração Inicial das Bases Importadas ( Clientes e Vendas)

In [ ]:
df_clientes.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- data_cadastro: string (nullable = true)



In [ ]:
df_vendas.printSchema()

root
 |-- venda_id: integer (nullable = true)
 |-- data_venda: date (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_unitario: double (nullable = true)



In [ ]:
df_clientes.show(5)
df_vendas.show(5)

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         5|    Guilherme Barros|davi-luizfarias@b...|   24|              Cuiaba| MT| Feminino|   20/04/2021|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
only showi

Seleção de Colunas

In [ ]:
df_clientes.select('nome','cidade').show(5)

+--------------------+--------------------+
|                nome|              cidade|
+--------------------+--------------------+
|        Miguel Porto|Sao Bernardo do C...|
|Gustavo Henrique ...|            Londrina|
|       Julia Correia|            Sao Luis|
|        Pietra Sales|              Recife|
|    Guilherme Barros|              Cuiaba|
+--------------------+--------------------+
only showing top 5 rows


In [ ]:
df_clientes.select('cidade')\
           .distinct()\
           .orderBy('cidade', ascending = True)\
           .show()

+------------------+
|            cidade|
+------------------+
|           Aracaju|
|Balneario Camboriu|
|             Belem|
|    Belo Horizonte|
|          Blumenau|
|          Brasilia|
|          Campinas|
|      Campo Grande|
|            Cuiaba|
|          Curitiba|
|     Florianopolis|
|         Fortaleza|
|           Goiania|
|         Guarulhos|
|       Joao Pessoa|
|         Joinville|
|           Jundiai|
|          Londrina|
|            Maceio|
|            Manaus|
+------------------+
only showing top 20 rows


In [ ]:
df_clientes.select(countDistinct('cidade')\
                   .alias("qtd.cidades"))\
                   .show()


+-----------+
|qtd.cidades|
+-----------+
|         39|
+-----------+



Seleção de Colunas Definindo parametro do Conjunto de colunas

In [ ]:
col_1 = list(set(df_clientes.columns)-{'email', 'idade', 'cliente_id'})

df_clientes2 = df_clientes\
   .select(*col_1)\
   .distinct()\
   .orderBy('cidade', ascending = True)\
   .show()

+-------+-------------+---+---------+--------------------+
| cidade|data_cadastro| UF|     sexo|                nome|
+-------+-------------+---+---------+--------------------+
|Aracaju|   28/02/2023| SE| Feminino|  Dra. Yasmin Taboao|
|Aracaju|   17/06/2022| SE| Feminino|     Rafaela Peixoto|
|Aracaju|   17/12/2024| SE|Masculino|      Melissa Farias|
|Aracaju|   20/12/2021| SE|Masculino|       Vitor da Cruz|
|Aracaju|   24/09/2020| SE|Masculino|    Vitória Carvalho|
|Aracaju|   25/10/2023| SE|Masculino|Marcos Vinicius B...|
|Aracaju|   27/04/2022| SE| Feminino|Sr. Guilherme Rez...|
|Aracaju|   04/11/2023| SE| Feminino|    Joaquim Ferreira|
|Aracaju|   29/01/2023| SE| Feminino|        Rafaela Melo|
|Aracaju|   18/08/2024| SE| Feminino| Luiz Felipe Almeida|
|Aracaju|   24/09/2020| SE| Feminino|        Paulo Mendes|
|Aracaju|   22/04/2024| SE| Feminino|       Vitor da Cruz|
|Aracaju|   09/10/2020| SE|Masculino|    Heloísa Oliveira|
|Aracaju|   24/11/2022| SE| Feminino|     Dra. Nina Cunh

Condição de Filtro com Where

In [ ]:
df_clientes.where("idade>=30" ).show()

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         6|Sr. Guilherme Rez...|cavalcantieloah@b...|   57|Sao Bernardo do C...| SP|Masculino|   13/08/2023|
|         8|Luiz Fernando Car...|joao-guilhermemor...|   55|       Florianopolis| SC| Feminino|   29/04/2021|
|         

In [ ]:
df_clientes.where("idade between 0 and 18" ).show()

+----------+--------------------+--------------+-----+--------------+---+---------+-------------+
|cliente_id|                nome|         email|idade|        cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------+-----+--------------+---+---------+-------------+
|       117|Sr. Emanuel da Co...|renan57@da.net|   18|      Sao Luis| MA|Masculino|   10/02/2020|
|       251|Sr. Emanuel da Co...|renan57@da.net|   18|Rio de Janeiro| RJ|Masculino|   14/09/2022|
|       276|Sr. Emanuel da Co...|renan57@da.net|   18|        Cuiaba| MT| Feminino|   10/09/2023|
|       794|Sr. Emanuel da Co...|renan57@da.net|   18|     Fortaleza| CE|Masculino|   30/06/2021|
+----------+--------------------+--------------+-----+--------------+---+---------+-------------+



Removendo Registros Duplicados

In [ ]:
df_clientes.count()

1000

In [ ]:
df_vendas.count()

1500

In [ ]:
df_vendas = df_vendas.dropDuplicates()
df_clientes = df_clientes.dropDuplicates()


Applied Analytcs - Responda os questionamentos

1 - Qual cidade gerou o maior valor total de venda?
2 - Qual produto mais vendido em quantidade e qual a sua média unitária?

In [ ]:
# Pergunta 01

df_join = df_vendas.join( df_clientes, on ="cliente_id", how = 'inner')
df_join.show(5)

+----------+--------+----------+-------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|cliente_id|venda_id|data_venda|produto|quantidade|valor_unitario|               nome|               email|idade|   cidade| UF|     sexo|data_cadastro|
+----------+--------+----------+-------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|        50|     785|2025-01-19| Tablet|         2|        580.64|    Bernardo Farias|       zpires@da.com|   49|  Jundiai| SP|Masculino|   10/10/2021|
|       154|     551|2025-01-04| Tablet|         4|       3969.83|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       154|     454|2025-01-29|Teclado|         4|        265.91|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       313|    1020|2025-02-20|Teclado|         1|       1757.54|João Vitor Monteiro|da

In [ ]:
cidade_maior_venda = df_join\
      .groupBy('cidade')\
      .agg(F.round(F.sum(F.col('quantidade')* F.col('valor_unitario')),2).alias('total_venda'))\
      .orderBy('total_venda', ascending = False)\
      .show(1)


+-------+-----------+
| cidade|total_venda|
+-------+-----------+
|Niteroi|  502446.27|
+-------+-----------+
only showing top 1 row
